# Módulo 02 · Lista de Exercícios

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## Como usar esta lista

Git só entra na cabeça pelos dedos. Ler sobre `rebase` não ensina nada; provocar um conflito e resolvê-lo, sim.

**Duas formas de resolver:**

1. **No terminal (recomendado)** — abra o terminal integrado do VS Code (`Ctrl+'`), navegue até a pasta do exercício e digite os comandos de verdade. É assim que você vai usar Git pelo resto da carreira.
2. **No notebook** — use a função `git(...)` fornecida abaixo. Cômodo para não trocar de janela, mas você não treina a digitação.

> 💡 **Sugestão:** faça os níveis 1 e 2 no notebook para pegar o ritmo, e os níveis 3 em diante no terminal.

| Nível | Aula base | Exercícios |
|-------|-----------|------------|
| 1 | 02_01 — Fundamentos | 1–10 |
| 2 | 02_02 — Branches e Remotos | 11–22 |
| 3 | 02_03 — Desfazendo | 23–32 |
| ⭐ | Cenários integrados | 33–40 |
| 🏆 | Projeto | Versionando o Atlas |

Execute a célula abaixo para preparar o ambiente.

In [ ]:
import shutil
import subprocess
from pathlib import Path

EXERCICIOS = Path("exercicios_git").resolve()
shutil.rmtree(EXERCICIOS, ignore_errors=True)
EXERCICIOS.mkdir(parents=True)

ATUAL = {"repo": None}


def novo_repo(nome, bare=False):
    """Cria um repositório limpo e o torna o repositório 'atual'."""
    caminho = EXERCICIOS / nome
    shutil.rmtree(caminho, ignore_errors=True)
    caminho.mkdir(parents=True)
    args = ["git", "init", "-b", "main"] + (["--bare"] if bare else [])
    subprocess.run(args + [str(caminho)], capture_output=True, text=True)
    if not bare:
        subprocess.run(["git", "config", "user.name", "Aluno Atlas"], cwd=caminho, capture_output=True)
        subprocess.run(["git", "config", "user.email", "aluno@aurora.com.br"], cwd=caminho, capture_output=True)
        ATUAL["repo"] = caminho
    print(f"📁 Repositório '{nome}' criado em {caminho}")
    return caminho


def git(*args, cwd=None, mostrar=True):
    """Executa um comando git no repositório atual."""
    destino = cwd or ATUAL["repo"]
    if destino is None:
        raise RuntimeError("Nenhum repositório ativo. Chame novo_repo(...) antes.")
    r = subprocess.run(["git", *args], cwd=destino, capture_output=True, text=True)
    if mostrar:
        print("$ git " + " ".join(args))
        saida = (r.stdout + r.stderr).rstrip()
        print(saida if saida else "(sem saída)")
        print()
    return r


def escrever(nome, conteudo, cwd=None):
    """Cria/sobrescreve um arquivo no repositório atual."""
    destino = Path(cwd or ATUAL["repo"]) / nome
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_text(conteudo, encoding="utf-8")
    print(f"✏️  {nome}")


def ler(nome, cwd=None):
    return (Path(cwd or ATUAL["repo"]) / nome).read_text(encoding="utf-8")


def commit(mensagem, cwd=None):
    """Atalho: add . + commit."""
    git("add", ".", cwd=cwd, mostrar=False)
    return git("commit", "-m", mensagem, cwd=cwd, mostrar=False)


def historico(n=10, cwd=None):
    git("log", f"-{n}", "--oneline", "--graph", "--all", cwd=cwd)


print("✅ Ambiente pronto.")
print()
print("Funções disponíveis:")
print("  novo_repo(nome)        — cria e ativa um repositório limpo")
print("  git(*args)             — executa um comando git")
print("  escrever(nome, texto)  — cria um arquivo")
print("  ler(nome)              — lê um arquivo")
print("  commit(mensagem)       — atalho para add . + commit")
print("  historico(n)           — log resumido com grafo")

---
# NÍVEL 1 — Fundamentos
*Aula `02_01`: init, status, add, commit, log, diff, .gitignore*

### 1. Primeiro repositório

Crie um repositório chamado `ex01`, configure `user.name` e `user.email` locais, crie um `README.md` com o título do projeto e faça o primeiro commit com uma mensagem no padrão Conventional Commits.

Depois confirme com `git log` que o commit tem seu nome e a mensagem correta.

In [ ]:
# 1.
novo_repo("ex01")

### 2. Os três estados

Crie três arquivos: `a.py`, `b.py` e `c.py`.

- Adicione **apenas** `a.py` ao stage
- Deixe `b.py` untracked
- Commite `a.py`, depois **edite-o** de novo

Rode `git status` e, em um comentário, descreva o estado de cada um dos três arquivos usando os termos corretos (*untracked*, *staged*, *modified*, *committed*).

In [ ]:
# 2.
novo_repo("ex02")

### 3. Lendo diffs

Crie um arquivo com 10 linhas e commite. Depois:

a) Altere a linha 3 e rode `git diff`. Quantas linhas aparecem com `-` e quantas com `+`? Por quê?
b) Adicione uma linha nova no fim. Rode `git diff` novamente.
c) Dê `git add` e rode `git diff` e `git diff --staged`. Explique a diferença.
d) Interprete o cabeçalho `@@ -x,y +a,b @@` do seu diff.

In [ ]:
# 3.
novo_repo("ex03")

### 4. Staging seletivo

Crie dois arquivos: `calculo.py` (uma correção de bug) e `relatorio.py` (uma funcionalidade nova).

Faça **dois commits separados**, cada um com sua mensagem apropriada. Não use `git add .`.

Confirme com `git log --stat` que cada commit contém apenas o arquivo certo.

In [ ]:
# 4.
novo_repo("ex04")

### 5. `.gitignore` com exceção

Escreva um `.gitignore` que:

- Ignore todos os `*.log`
- **Exceto** `importante.log`
- Ignore a pasta `temp/` inteira
- Ignore `.env`, mas **não** `.env.example`

Crie todos esses arquivos e prove com `git status` que o comportamento está correto.

In [ ]:
# 5.
novo_repo("ex05")

### 6. A armadilha do `.gitignore`

a) Crie e **commite** um arquivo `segredo.txt`.
b) Só depois adicione `segredo.txt` ao `.gitignore`.
c) Modifique o arquivo. O Git ainda vê a mudança? Por quê?
d) Corrija a situação mantendo o arquivo no disco.
e) Confirme com `git ls-files` que ele não é mais rastreado.

In [ ]:
# 6.
novo_repo("ex06")

### 7. Explorando o `git log`

Crie 6 commits, alternando entre dois arquivos e usando tipos variados (`feat`, `fix`, `docs`, `chore`).

Depois responda usando o comando adequado:

a) Quais commits mexeram em `metricas.py`?
b) Quais commits têm "frete" na mensagem?
c) Em qual commit a string `calcular_frete` foi **introduzida** no código?
d) Quantas linhas cada commit adicionou/removeu?

In [ ]:
# 7.
novo_repo("ex07")

### 8. Mensagens de commit

Reescreva no padrão Conventional Commits, e em um comentário justifique o tipo escolhido:

| # | Mensagem original |
|---|-------------------|
| a | `mudanças` |
| b | `arrumei o bug` |
| c | `agora o relatorio ta funcionando e tambem mudei o readme` |
| d | `atualizei as bibliotecas` |
| e | `refatorei tudo` |
| f | `adicionando testes` |

**Dica para (c):** se uma mensagem precisa de "e também", provavelmente deveriam ser dois commits.

<!-- 8. — suas respostas aqui -->

### 9. Renomear e remover

a) Crie e commite `relatorio_vendas.py`.
b) Renomeie para `relatorio.py` usando `git mv` e commite.
c) Verifique com `git log --stat -1` como o Git registrou a operação.
d) Crie e commite `obsoleto.py`. Depois remova-o com `git rm` e commite.
e) Crie `local.ini`, commite, e depois remova só do índice mantendo no disco.

In [ ]:
# 9.
novo_repo("ex09")

### 10. Anatomia do commit

Faça um commit e depois use `git show` e `git cat-file` para responder:

a) Qual o hash completo?
b) Quem é o autor e qual o timestamp?
c) Qual o hash do commit pai?
d) O que `git cat-file -p <hash>` mostra?

**Dica:** `git log -1 --format=%H` dá o hash completo; `git cat-file -p <hash>` mostra o objeto cru.

In [ ]:
# 10.
novo_repo("ex10")

---
# NÍVEL 2 — Branches e Remotos
*Aula `02_02`: branch, switch, merge, conflitos, remote, push, pull*

### 11. Fast-forward

Crie um repositório com 2 commits em `main`. Depois:

a) Crie `feature/calculo` e faça 2 commits nele
b) Volte para `main` e mescle
c) Confirme com `git log --graph` que **não** houve commit de merge
d) Explique por quê

In [ ]:
# 11.
novo_repo("ex11")

### 12. Merge de 3 vias

Mesmo cenário do exercício 11, mas **faça também um commit em `main`** antes de mesclar.

a) O merge continua sendo fast-forward? Por quê?
b) Quantos pais tem o commit de merge? Prove com `git log -1 --format=%p`.
c) Desenhe (em comentário) o grafo resultante.

In [ ]:
# 12.
novo_repo("ex12")

### 13. Merge sem conflito

Crie dois branches a partir de `main`. Em cada um, altere um **arquivo diferente**. Mescle os dois em `main`.

Não deve haver conflito. Explique, em um comentário, por que o Git conseguiu resolver sozinho.

In [ ]:
# 13.
novo_repo("ex13")

### 14. Merge com mudanças no mesmo arquivo, linhas diferentes

Agora dois branches alterando **o mesmo arquivo**, mas em **linhas bem distantes** (por exemplo, uma no topo e outra no fim de um arquivo de 30 linhas).

Ainda não deve conflitar. Por quê? O que determina se o Git consegue mesclar automaticamente?

In [ ]:
# 14.
novo_repo("ex14")

### 15. Conflito — resolvendo de três formas

Provoque um conflito real: dois branches alterando **a mesma linha**.

Resolva de três maneiras diferentes, usando `git merge --abort` entre as tentativas:

a) Ficando com a versão de `main`
b) Ficando com a versão do branch que entra
c) Combinando as duas

Em cada caso, confirme o resultado final do arquivo.

In [ ]:
# 15.
novo_repo("ex15")

### 16. Conflito em múltiplos arquivos

Provoque um conflito em **três arquivos ao mesmo tempo**.

a) Como `git status` apresenta isso?
b) Resolva um arquivo, dê `git add`, e rode `git status` de novo. O que mudou?
c) É possível commitar com apenas um resolvido? O que o Git diz?

In [ ]:
# 16.
novo_repo("ex16")

### 17. `--no-ff` e `--squash`

Crie um branch com 4 commits e mescle-o em `main` de três formas diferentes (recriando o cenário a cada vez):

a) Merge padrão
b) `git merge --no-ff`
c) `git merge --squash`

Compare os três `git log --graph`. Quando você usaria cada um?

In [ ]:
# 17.
novo_repo("ex17")

### 18. Limpeza de branches

Crie 5 branches. Mescle 3 deles em `main`.

a) Use `git branch --merged` e `git branch --no-merged`
b) Apague os mesclados com `-d`
c) Tente apagar um não mesclado com `-d`. O que acontece?
d) Force com `-D`. Quando isso é aceitável e quando é perigoso?

In [ ]:
# 18.
novo_repo("ex18")

### 19. Repositório remoto local

a) Crie um repositório **bare** chamado `servidor.git`
b) Crie um repositório normal, adicione o bare como `origin`
c) Faça commits e `push -u origin main`
d) Confirme o vínculo com `git branch -vv`
e) Inspecione o conteúdo do bare — por que não há arquivos do projeto lá?

In [ ]:
# 19.
servidor = novo_repo("servidor.git", bare=True)
novo_repo("ex19")

### 20. Clone e sincronização

Continuando do exercício 19:

a) Clone o `servidor.git` em `clone_a` e `clone_b`
b) Faça um commit em `clone_a` e envie
c) Em `clone_b`, use `git fetch` — o arquivo apareceu no disco?
d) Use `git log origin/main` para inspecionar o que veio
e) Agora dê `git pull` e confirme

Explique a diferença prática entre `fetch` e `pull`.

In [ ]:
# 20.

### 21. Push rejeitado

Continuando:

a) Faça commits **divergentes** em `clone_a` e `clone_b`
b) Envie o de `clone_a`
c) Tente enviar o de `clone_b` — leia a mensagem de erro com atenção
d) Resolva corretamente
e) Explique por que `git push --force` seria a resposta errada aqui, e o que `--force-with-lease` faria de diferente

In [ ]:
# 21.

### 22. Branch remoto

a) Em `clone_a`, crie `feature/nova` e envie com `-u`
b) Em `clone_b`, rode `git fetch` e depois `git branch -a`
c) Crie um branch local a partir do remoto e trabalhe nele
d) Apague o branch remoto com `git push -d origin feature/nova`
e) Em `clone_b`, limpe as referências obsoletas com `git fetch --prune`

In [ ]:
# 22.

---
# NÍVEL 3 — Desfazendo
*Aula `02_03`: restore, amend, revert, reset, stash, reflog*

### 23. `restore` em três situações

Para cada caso abaixo, identifique o comando correto e execute:

a) Editei `config.py` e quero descartar (não dei `add`)
b) Dei `add` em `config.py` e quero tirar do stage, mantendo a edição
c) Dei `add` e quero tirar do stage **e** descartar a edição
d) Quero o `config.py` como ele era 3 commits atrás

In [ ]:
# 23.
novo_repo("ex23")

### 24. `--amend`

a) Faça um commit com a mensagem `"asdfgh"`
b) Corrija a mensagem com `--amend`
c) Compare os hashes antes e depois — o que aconteceu com o commit original?
d) Esqueça um arquivo, adicione-o ao commit com `--amend --no-edit`
e) Em que situação você **não** poderia usar `--amend`?

In [ ]:
# 24.
novo_repo("ex24")

### 25. Os três modos do `reset`

Crie 4 commits. Depois, recriando o cenário a cada vez, teste os três modos voltando 2 commits:

| Modo | Histórico | Stage | Working dir |
|------|-----------|-------|-------------|
| `--soft` | ? | ? | ? |
| `--mixed` | ? | ? | ? |
| `--hard` | ? | ? | ? |

Preencha a tabela **observando** o resultado de `git log --oneline` e `git status --short` após cada um.

In [ ]:
# 25.
novo_repo("ex25")

### 26. Squash local com `reset --soft`

Faça 6 commits com mensagens ruins (`wip`, `wip2`, `teste`, ...).

Junte todos em **um** commit bem escrito usando `reset --soft`.

Confirme que:

a) O histórico tem 1 commit no lugar dos 6
b) Nenhuma linha de código se perdeu (compare o conteúdo dos arquivos)

In [ ]:
# 26.
novo_repo("ex26")

### 27. `revert`

a) Faça 3 commits
b) Reverta o **do meio** (não o último)
c) Quantos commits o histórico tem agora? Por quê?
d) O conteúdo do arquivo voltou ao esperado?
e) Reverta o próprio revert. O que acontece?

In [ ]:
# 27.
novo_repo("ex27")

### 28. `reset` vs `revert` — a decisão

Para cada cenário, escolha `reset` ou `revert` e justifique em uma frase:

| # | Cenário | Comando | Por quê |
|---|---------|---------|---------|
| a | Commit local, ainda não enviado, mensagem errada | | |
| b | Commit já no GitHub, quebrou produção | | |
| c | Quero juntar meus 5 commits antes do PR | | |
| d | Commit num branch compartilhado com 3 pessoas | | |
| e | Commitei arquivo errado, ninguém puxou ainda | | |

<!-- 28. — preencha a tabela aqui -->

### 29. `stash`

a) Comece uma alteração e crie um arquivo novo (untracked)
b) Guarde tudo com `git stash -u -m "descrição"`
c) Confirme que o diretório ficou limpo (inclusive o untracked)
d) Faça um commit não relacionado
e) Recupere com `pop`
f) Agora crie **dois** stashes. Como aplicar o **primeiro**?
g) Qual a diferença entre `pop` e `apply`?

In [ ]:
# 29.
novo_repo("ex29")

### 30. `reflog` — recuperação

a) Faça 5 commits
b) Apague os 3 últimos com `reset --hard`
c) Confirme que sumiram do `git log`
d) Encontre-os no `reflog`
e) Recupere **sem** mexer no branch atual (crie um branch de recuperação)
f) Confirme que os 3 commits estão lá

In [ ]:
# 30.
novo_repo("ex30")

### 31. `cherry-pick`

a) Crie um branch com 4 commits
b) Traga **apenas o segundo** para `main`
c) Compare o hash original com o hash do commit criado em `main` — são iguais?
d) Traga também o quarto. O que acontece se ele depender do terceiro?

In [ ]:
# 31.
novo_repo("ex31")

### 32. `clean`

a) Crie arquivos untracked e uma pasta untracked com arquivos dentro
b) Rode `git clean -n` — o que ele lista?
c) Rode `git clean -f` — a pasta foi removida?
d) Use a flag correta para remover também pastas
e) Explique por que `git clean -fdx` é perigoso em um projeto Python

In [ ]:
# 32.
novo_repo("ex32")

---
# ⭐ NÍVEL INTEGRADO — Cenários reais

Cada exercício é uma situação que você vai viver. Resolva do começo ao fim.

### 33. A senha vazada

**Cenário:** você commitou um `.env` com a senha do banco de produção. O commit ainda **não** foi enviado ao GitHub.

a) Reproduza a situação
b) Remova o arquivo do histórico
c) Adicione ao `.gitignore`
d) Confirme que o arquivo continua no disco mas não é rastreado
e) **Em comentário:** o que mudaria se o commit já tivesse sido enviado? Qual seria o **primeiro** passo?

In [ ]:
# 33.
novo_repo("ex33")

### 34. Trabalhei no branch errado

**Cenário:** você fez 3 commits direto em `main`. Eles deveriam estar em `feature/relatorio`.

a) Reproduza
b) Mova os 3 commits para um branch novo, sem perder nada
c) Deixe `main` como estava antes
d) Confirme com `git log --graph --all`

In [ ]:
# 34.
novo_repo("ex34")

### 35. Hotfix no meio da feature

**Cenário:** você está no meio de uma feature (código pela metade, não commitável) e chega um bug crítico em produção.

a) Reproduza o estado "pela metade"
b) Guarde o trabalho
c) Vá para `main`, crie `hotfix/calculo-frete`, corrija e mescle
d) Volte para a feature e recupere o trabalho
e) Confirme que a correção está em `main` e a feature continua incompleta no seu branch

In [ ]:
# 35.
novo_repo("ex35")

### 36. Bisect — encontrando o commit culpado

**Cenário:** o cálculo de faturamento está errado, mas há 12 commits desde a última vez que funcionou.

O `git bisect` faz busca binária pelo commit que introduziu o problema — em 12 commits, encontra em ~4 tentativas.

a) Crie 12 commits, e no 7º introduza um bug (por exemplo, remover o filtro de status)
b) Use `git bisect start`, `git bisect bad`, `git bisect good <hash-antigo>`
c) A cada passo, verifique se o código está certo e marque `good` ou `bad`
d) Encontre o culpado e encerre com `git bisect reset`

**Dica:** dá para automatizar com `git bisect run python teste.py` — o script deve sair com código 0 (bom) ou diferente de 0 (ruim).

In [ ]:
# 36.
novo_repo("ex36")

### 37. Colaboração completa

**Cenário:** simule um time de 3 pessoas com um repositório remoto.

a) Crie o bare `origin.git` e três clones (`ana`, `bruno`, `carla`)
b) Ana cria a estrutura inicial e envia
c) Bruno e Carla sincronizam
d) Cada um cria um branch de feature e faz commits
e) Bruno mescla primeiro
f) Carla precisa sincronizar antes de mesclar — e conflita
g) Carla resolve, mescla e envia
h) Ana sincroniza e recebe tudo
i) Confirme que os três têm o mesmo histórico (`git log --oneline` idêntico)

In [ ]:
# 37.

### 38. Recuperação após desastre múltiplo

**Cenário:** um estagiário rodou, em sequência:

```bash
git reset --hard HEAD~5
git clean -fd
git branch -D feature/importante
```

a) Reproduza o cenário (crie 5 commits, arquivos untracked e um branch)
b) Execute os três comandos
c) Recupere o **máximo** possível
d) **Em comentário:** o que é irrecuperável, e por quê?

In [ ]:
# 38.
novo_repo("ex38")

### 39. Arqueologia

**Cenário:** o código tem uma função `calcular_desconto` que ninguém sabe de onde veio.

a) Crie um histórico de 8 commits onde essa função nasce no 3º, é alterada no 5º e quase removida no 7º
b) Use `git log -S "calcular_desconto"` para achar quando ela apareceu
c) Use `git log -p -- arquivo.py` para ver a evolução
d) Use `git blame arquivo.py` para ver quem escreveu cada linha
e) Use `git show <hash>` para ver o commit completo

In [ ]:
# 39.
novo_repo("ex39")

### 40. Automação com aliases e hooks

a) Configure estes aliases e teste cada um:

```bash
git config --global alias.st "status --short"
git config --global alias.lg "log --oneline --graph --all --decorate"
git config --global alias.last "log -1 --stat"
git config --global alias.unstage "restore --staged"
```

b) Crie um **hook** `pre-commit` em `.git/hooks/pre-commit` que impeça o commit se algum arquivo `.py` contiver a string `TODO: REMOVER`.

**Dica:** o hook é um script executável. Em Python:

```python
#!/usr/bin/env python3
import subprocess, sys
r = subprocess.run(["git", "diff", "--cached", "--name-only"],
                   capture_output=True, text=True)
for arquivo in r.stdout.split():
    if arquivo.endswith(".py"):
        if "TODO: REMOVER" in open(arquivo, encoding="utf-8").read():
            print(f"❌ {arquivo} contém 'TODO: REMOVER'")
            sys.exit(1)
sys.exit(0)
```

No Linux/macOS, dê permissão: `chmod +x .git/hooks/pre-commit`.

c) Teste: crie um arquivo com a marca e tente commitar.

In [ ]:
# 40.
novo_repo("ex40")

---
---

# 🏆 PROJETO DO MÓDULO — Versionando o Atlas

## Contexto

> *"Ontem o estagiário salvou por cima do `relatorio_vendas.py`. Perdemos o dia inteiro. Isso não pode acontecer de novo."*
> — Diretora Comercial

Você entregou o relatório do Módulo 01 e ele funciona. Agora ele precisa parar de viver em uma pasta solta no seu computador.

## Objetivo

Colocar o `projeto_Atlas` sob versionamento, com histórico limpo, publicado no GitHub, e com automações que reduzem trabalho manual.

## Parte A — Repositório local

1. Inicialize o Git em `projeto_Atlas/` com branch `main`
2. Crie um `.gitignore` adequado a um projeto Python (use [gitignore.io](https://www.toptal.com/developers/gitignore) combinando Python + VisualStudioCode + seu SO)
3. **Antes do primeiro commit**, rode `git status` e confirme que:
   - `.venv/` não aparece
   - `__pycache__/` não aparece
   - `saida/` não aparece
   - `dados/brutos/*.csv` **aparece** (são dados de exemplo pequenos — versionamos)
4. Faça o primeiro commit

## Parte B — Histórico com significado

Em vez de um commit gigante com tudo, **construa o histórico como se você tivesse desenvolvido em etapas**. Faça pelo menos 8 commits, cada um com uma mensagem no padrão Conventional Commits:

| # | Commit | Conteúdo |
|---|--------|----------|
| 1 | `chore: estrutura inicial do projeto` | `.gitignore`, `README.md`, pastas |
| 2 | `chore(dados): adiciona CSVs de exemplo` | `dados/brutos/` |
| 3 | `feat(config): adiciona constantes e caminhos` | `config.py` |
| 4 | `feat: adiciona exceções de domínio` | `excecoes.py` |
| 5 | `feat(formatacao): adiciona formatação BRL` | `formatacao.py` |
| 6 | `feat(leitura): adiciona leitor de CSV` | `leitura.py` |
| 7 | `feat(validacao): adiciona validação de linhas` | `validacao.py` |
| 8 | `feat(metricas): adiciona cálculo de métricas` | `metricas.py` |
| 9 | `feat(relatorios): adiciona renderização txt e json` | `relatorios.py` |
| 10 | `feat(cli): adiciona ponto de entrada` | `cli.py`, `main.py` |
| 11 | `docs: adiciona README com instruções de setup` | `README.md` completo |

**Dica:** você pode usar `git add <arquivo>` seletivo para construir esse histórico a partir do que já existe.

## Parte C — Branches

1. Crie `feature/relatorio-por-canal`
2. Implemente o agrupamento por canal de venda
3. Commite em pelo menos 2 commits
4. Volte para `main` e faça uma alteração no `README.md`
5. Mescle a feature (agora será um merge de 3 vias)
6. Apague o branch

## Parte D — GitHub

1. Crie o repositório `atlas` no GitHub (privado ou público, você escolhe)
2. Configure autenticação por **SSH** (gere a chave, cadastre, teste com `ssh -T git@github.com`)
3. Conecte e envie:
   ```bash
   git remote add origin git@github.com:seu-usuario/atlas.git
   git push -u origin main
   ```
4. Confirme no navegador que o histórico apareceu

## Parte E — Automações shell

Crie uma pasta `scripts/` com automações. **Faça as duas versões** (`.sh` para Linux/macOS e `.ps1` para Windows) ou apenas a do seu sistema.

| Script | O que faz |
|--------|-----------|
| `setup.sh` / `setup.ps1` | Cria o venv, ativa e instala dependências |
| `rodar.sh` / `rodar.ps1` | Executa o relatório com o CSV padrão |
| `limpar.sh` / `limpar.ps1` | Apaga `saida/`, `__pycache__/` e caches |
| `verificar.sh` / `verificar.ps1` | Roda o relatório contra o CSV limpo **e** o sujo, e reporta |

Os esqueletos estão em `projeto_Atlas/scripts/` — com comentários indicando o que implementar.

## Parte F — Simulação de desastre

Prove que você sabe se recuperar. Documente cada passo em `docs/RECUPERACAO.md`:

1. Faça 3 commits
2. Apague-os com `git reset --hard HEAD~3`
3. Recupere-os pelo reflog
4. Faça um commit que "quebra" o código
5. Reverta com `git revert`
6. Registre no documento os comandos exatos que usou

## Critérios de avaliação

| Critério | Peso |
|----------|------|
| `.gitignore` correto (nada de `.venv`, `__pycache__`, `.env`) | 20% |
| Histórico com commits atômicos e mensagens no padrão | 30% |
| Branch criado, usado e mesclado corretamente | 15% |
| Repositório publicado no GitHub via SSH | 15% |
| Scripts de automação funcionando | 10% |
| `docs/RECUPERACAO.md` documentando os cenários | 10% |

## Desafios extras

- ⭐ Configure um hook `pre-commit` que rode `ruff check` e barre o commit se houver erro
- ⭐ Crie uma tag anotada `v0.1.0` e envie: `git tag -a v0.1.0 -m "Atlas M01"` + `git push --tags`
- ⭐ Escreva um `CONTRIBUTING.md` explicando o padrão de branches e commits do projeto
- ⭐⭐ Configure um template de commit (`git config commit.template .gitmessage`)
- ⭐⭐ Abra um Pull Request de verdade no seu próprio repositório e faça a autorrevisão

## Entrega

Ao terminar, você deve conseguir mandar o link do repositório para alguém e essa pessoa deve conseguir:

```bash
git clone git@github.com:seu-usuario/atlas.git
cd atlas
./scripts/setup.sh
./scripts/rodar.sh
```

e ver o relatório rodando.

In [ ]:
# 🏆 Espaço de trabalho para o projeto.
# Sugestão: faça as partes A–D no TERMINAL, dentro de projeto_Atlas/.
# Use esta célula só para conferências rápidas, por exemplo:
#
#     !git -C ../projeto_Atlas log --oneline --graph --all
#     !git -C ../projeto_Atlas status --short

---

## ✅ Autoavaliação do Módulo 02

**Fundamentos**

- [ ] Explico o modelo de snapshots e as três áreas
- [ ] Uso `git status` reflexivamente
- [ ] Escrevo commits atômicos com mensagens no padrão
- [ ] Sei o que nunca deve ser versionado

**Branches**

- [ ] Crio, troco e apago branches sem hesitar
- [ ] Diferencio fast-forward de merge de 3 vias
- [ ] Resolvo conflitos sem pânico
- [ ] Sei que `git merge --abort` existe

**Colaboração**

- [ ] Configurei SSH e publico no GitHub
- [ ] Diferencio `fetch` de `pull`
- [ ] Sei por que `push --force` é perigoso
- [ ] Sei abrir e revisar um Pull Request

**Recuperação**

- [ ] Escolho entre `restore`, `reset`, `revert` e `stash` conscientemente
- [ ] Uso `reflog` quando algo dá errado
- [ ] Sei o que é irrecuperável (e por isso commito com frequência)

**Autonomia**

- [ ] Não tenho medo de experimentar, porque sei desfazer
- [ ] Meu histórico conta a história do projeto, não o meu processo de tentativa e erro

---

### ➡️ Próximo módulo

**Módulo 03 — SQL.** Dor da Aurora: *"Os dados estão em 14 planilhas diferentes."* Você vai modelar um schema relacional e migrar o Atlas de CSV para banco de dados.